# 🇪🇺 EU Quality of Life Index: Where Does Ireland Rank?

**Data Analysis and Visualisation CA1 — Composite Indicator Generation and Visualisation**  

This notebook develops a composite indicator to compare the quality of life of the 27 European Union countries using Eurostat data.

---

## Project aim

The aim of this project is to construct an **EU Quality of Life Index** by combining indicators related to economic wellbeing, health, education, employment, and subjective wellbeing. The final index produces a single score for each EU country, allowing the countries to be ranked and compared. A specific focus is placed on **Ireland's position within the EU**.

---



In [ ]:


import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)


CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data").exists():
    BASE_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    BASE_DIR = CURRENT_DIR.parent
else:
    raise FileNotFoundError("Could not find the data folder. Run this notebook from the project root, code folder, or notebooks folder.")

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
TABLES_DIR = OUTPUT_DIR / "tables"
CHARTS_DIR = OUTPUT_DIR / "charts"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Data directory:", DATA_DIR)
print("Data directory exists:", DATA_DIR.exists())
print("Charts output:", CHARTS_DIR)
print("Tables output:", TABLES_DIR)


Base directory: /Users/mohamedashiks/Desktop/CA1_Quality_of_Life_Index
Data directory: /Users/mohamedashiks/Desktop/CA1_Quality_of_Life_Index/data
Data directory exists: True
Charts output: /Users/mohamedashiks/Desktop/CA1_Quality_of_Life_Index/output/charts
Tables output: /Users/mohamedashiks/Desktop/CA1_Quality_of_Life_Index/output/tables


# Define EU countries, indicators, and weights

The weights are deliberately transparent and easy to justify. Economic and health conditions receive the largest weights because they are core structural dimensions of quality of life. Education and wellbeing are also important and receive substantial weights. Employment receives a smaller but still meaningful weight because it captures labour-market conditions.

In [25]:
EU_COUNTRIES = [
    "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czechia",
    "Denmark", "Estonia", "Finland", "France", "Germany", "Greece",
    "Hungary", "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg",
    "Malta", "Netherlands", "Poland", "Portugal", "Romania", "Slovakia",
    "Slovenia", "Spain", "Sweden"
]

RAW_FEATURES = [
    "GDP_PPS",
    "Life_Expectancy",
    "Education_Expenditure",
    "Unemployment_Rate",
    "High_Life_Satisfaction"
]

WEIGHTS = {
    "Economic_Subindex": 0.25,
    "Health_Subindex": 0.25,
    "Education_Subindex": 0.20,
    "Wellbeing_Subindex": 0.20,
    "Employment_Subindex": 0.10
}

weights_df = pd.DataFrame({
    "Subindex": list(WEIGHTS.keys()),
    "Weight": list(WEIGHTS.values())
})

weights_df.to_csv(TABLES_DIR / "index_weights.csv", index=False)
weights_df

,Subindex,Weight
0,Economic_Subindex,0.25
1,Health_Subindex,0.25
2,Education_Subindex,0.20
3,Wellbeing_Subindex,0.20
4,Employment_Subindex,0.10


# Load and clean raw Eurostat datasets

The function below selects the latest available value for each country up to 2023. This creates one row per country for each indicator.

In [ ]:
def latest_value_by_country(df, value_name):
    """Return latest non-missing value for each EU country."""
    df = df[df["geo"].isin(EU_COUNTRIES)].copy()
    df = df.dropna(subset=["OBS_VALUE"])
    df = df.sort_values(["geo", "TIME_PERIOD"])
    latest = df.groupby("geo").tail(1)
    latest = latest[["geo", "TIME_PERIOD", "OBS_VALUE"]]
    latest = latest.rename(columns={
        "geo": "Country",
        "TIME_PERIOD": f"{value_name}_Year",
        "OBS_VALUE": value_name
    })
    return latest


gdp = pd.read_csv(DATA_DIR / "GDP per capita in PPS.csv")
life = pd.read_csv(DATA_DIR / "Life expectancy at birth by sex.csv")
education = pd.read_csv(DATA_DIR / "Public expenditure on education in current prices, by education level and programme orientation.csv")
unemployment = pd.read_csv(DATA_DIR / "Unemployment rates by citizenship.csv")
happiness = pd.read_csv(DATA_DIR / "Overall life satisfaction by level of satisfaction, age and educational attainment.csv")

print("Raw file shapes:")
print("GDP:", gdp.shape)
print("Life expectancy:", life.shape)
print("Education:", education.shape)
print("Unemployment:", unemployment.shape)
print("Life satisfaction:", happiness.shape)

Raw file shapes:
GDP: (492, 10)
Life expectancy: (752, 11)
Education: (11863, 10)
Unemployment: (569851, 12)
Life satisfaction: (104904, 14)
